In [49]:
import json
from collections import Counter

class SimpleTokenizer:
    def __init__(self, vocab=None):
        self.vocab = vocab or {}
        self.word_to_id = {}
        self.id_to_word = {}

    def train(self, texts, min_freq=1, special_tokens=None):
        special_tokens = special_tokens or ["<pad>", "<unk>", "<bos>", "<eos>"]

        counter = Counter()
        for text in texts:
            counter.update(text.split())

        vocab_words = [word for word, freq in counter.items() if freq >= min_freq]
        vocab_words = special_tokens + sorted(vocab_words)

        self.word_to_id = {word: i for i, word in enumerate(vocab_words)}
        self.id_to_word = {i: word for word, i in self.word_to_id.items()}
        self.vocab = self.word_to_id

    def encode(self, text):
        tokens = text.split()
        unk_id = self.word_to_id["<unk>"]
        return [self.word_to_id.get(token, unk_id) for token in tokens]

    def decode(self, ids):
        return " ".join([self.id_to_word.get(i, "<unk>") for i in ids])

In [50]:
import re

def normalize_arabic(text):
    text = str(text)

    text = re.sub(r'ـ+', '', text)
    text = re.sub(r'[أإآ]', 'ا', text)
    text = re.sub(r'ى', 'ي', text)
    text = re.sub(r'ة', 'ه', text)
    text = re.sub(r'[A-Za-z]+', ' ', text)
    text = re.sub(r'[^\u0600-\u06FF0-9\s\.\،\:]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text

with open("../../data/pretrain/data.txt", "r", encoding="utf-8") as f:
    texts = f.readlines()

texts = [normalize_arabic(t) for t in texts if str(t).strip()]

tokenizer = SimpleTokenizer()
tokenizer.train(texts, min_freq=1)

print("Vocab size:", len(tokenizer.word_to_id))

sample_text = texts[0]
encoded = tokenizer.encode(sample_text)
decoded = tokenizer.decode(encoded[:20])

print("Sample text:", sample_text[:200])
print("Encoded:", encoded[:20])
print("Decoded:", decoded)

Vocab size: 22085
Sample text: دور للبيع في حي النرجس بمدينه الرياض. السعر 1500000 ريال. المساحه 116.0 متر مربع. عدد غرف النوم 4. عدد دورات المياه 3. عدد غرف المعيشه 1. المميزات: مفروش: لا، مكيف: لا، مصعد: لا، مدخل سياره: نعم، مسبح
Encoded: [13281, 16016, 15163, 12960, 10194, 11701, 8117, 8250, 1623, 13583, 9640, 1203, 16726, 17081, 14635, 14904, 10281, 3483, 14635, 13285]
Decoded: دور للبيع في حي النرجس بمدينه الرياض. السعر 1500000 ريال. المساحه 116.0 متر مربع. عدد غرف النوم 4. عدد دورات


In [51]:
class PretrainDataset:
    def __init__(self, token_ids, seq_length=32):
        self.token_ids = token_ids
        self.seq_length = seq_length
        self.inputs = []
        self.targets = []

        for i in range(len(token_ids) - seq_length):
            x = token_ids[i:i + seq_length]
            y = token_ids[i + 1:i + seq_length + 1]
            self.inputs.append(x)
            self.targets.append(y)

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        return self.inputs[idx], self.targets[idx]

In [52]:
all_text = " ".join(texts)
all_token_ids = tokenizer.encode(all_text)

print("Total token ids:", len(all_token_ids))
print("First 30 token ids:", all_token_ids[:30])

Total token ids: 675745
First 30 token ids: [13281, 16016, 15163, 12960, 10194, 11701, 8117, 8250, 1623, 13583, 9640, 1203, 16726, 17081, 14635, 14904, 10281, 3483, 14635, 13285, 10150, 2922, 14635, 14904, 9918, 919, 10023, 17648, 15578, 17737]


In [53]:
dataset = PretrainDataset(all_token_ids, seq_length=32)

print("Number of training samples:", len(dataset))

x, y = dataset[0]
print("Input ids:", x[:10])
print("Target ids:", y[:10])

print("Decoded input:", tokenizer.decode(x[:20]))
print("Decoded target:", tokenizer.decode(y[:20]))

Number of training samples: 675713
Input ids: [13281, 16016, 15163, 12960, 10194, 11701, 8117, 8250, 1623, 13583]
Target ids: [16016, 15163, 12960, 10194, 11701, 8117, 8250, 1623, 13583, 9640]
Decoded input: دور للبيع في حي النرجس بمدينه الرياض. السعر 1500000 ريال. المساحه 116.0 متر مربع. عدد غرف النوم 4. عدد دورات
Decoded target: للبيع في حي النرجس بمدينه الرياض. السعر 1500000 ريال. المساحه 116.0 متر مربع. عدد غرف النوم 4. عدد دورات المياه


In [54]:
import torch

X = torch.tensor(dataset.inputs, dtype=torch.long)
Y = torch.tensor(dataset.targets, dtype=torch.long)

print("X shape:", X.shape)
print("Y shape:", Y.shape)

X shape: torch.Size([675713, 32])
Y shape: torch.Size([675713, 32])


In [55]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

In [56]:
class SelfAttentionHead(nn.Module):
    def __init__(self, embed_dim, head_dim, block_size):
        super().__init__()
        self.key = nn.Linear(embed_dim, head_dim, bias=False)
        self.query = nn.Linear(embed_dim, head_dim, bias=False)
        self.value = nn.Linear(embed_dim, head_dim, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(0.1)

    def forward(self, x):
        B, T, C = x.shape

        k = self.key(x)      # (B, T, head_dim)
        q = self.query(x)    # (B, T, head_dim)

        wei = q @ k.transpose(-2, -1) / math.sqrt(k.size(-1))   # (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)

        v = self.value(x)    # (B, T, head_dim)
        out = wei @ v        # (B, T, head_dim)
        return out

In [57]:
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, embed_dim, block_size):
        super().__init__()
        head_dim = embed_dim // num_heads
        self.heads = nn.ModuleList([
            SelfAttentionHead(embed_dim, head_dim, block_size)
            for _ in range(num_heads)
        ])
        self.proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(0.1)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

In [58]:
class FeedForward(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embed_dim, 4 * embed_dim),
            nn.ReLU(),
            nn.Linear(4 * embed_dim, embed_dim),
            nn.Dropout(0.1)
        )

    def forward(self, x):
        return self.net(x)

In [59]:
class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, block_size):
        super().__init__()
        self.sa = MultiHeadAttention(num_heads, embed_dim, block_size)
        self.ffwd = FeedForward(embed_dim)
        self.ln1 = nn.LayerNorm(embed_dim)
        self.ln2 = nn.LayerNorm(embed_dim)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

In [60]:
class TinyGPT(nn.Module):
    def __init__(self, vocab_size, block_size, embed_dim=128, num_heads=4, num_layers=2):
        super().__init__()
        self.block_size = block_size

        self.token_embedding = nn.Embedding(vocab_size, embed_dim)
        self.position_embedding = nn.Embedding(block_size, embed_dim)

        self.blocks = nn.Sequential(*[
            TransformerBlock(embed_dim, num_heads, block_size)
            for _ in range(num_layers)
        ])

        self.ln_f = nn.LayerNorm(embed_dim)
        self.lm_head = nn.Linear(embed_dim, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.token_embedding(idx)                       # (B, T, C)
        pos_emb = self.position_embedding(torch.arange(T, device=idx.device))  # (T, C)
        x = tok_emb + pos_emb                                     # (B, T, C)

        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)                                  # (B, T, vocab_size)

        loss = None
        if targets is not None:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

In [61]:
split_idx = int(0.9 * len(X))

X_train = X[:split_idx]
Y_train = Y[:split_idx]

X_val = X[split_idx:]
Y_val = Y[split_idx:]

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)

X_train: torch.Size([608141, 32])
X_val: torch.Size([67572, 32])


In [62]:
split_idx = int(0.9 * len(X))

X_train, X_val = X[:split_idx], X[split_idx:]
Y_train, Y_val = Y[:split_idx], Y[split_idx:]

print("X_train:", X_train.shape)
print("Y_train:", Y_train.shape)
print("X_val:", X_val.shape)
print("Y_val:", Y_val.shape)

# smaller subsets for faster notebook training
X_train_small = X_train[:50000]
Y_train_small = Y_train[:50000]

X_val_small = X_val[:10000]
Y_val_small = Y_val[:10000]

print("X_train_small:", X_train_small.shape)
print("X_val_small:", X_val_small.shape)

X_train: torch.Size([608141, 32])
Y_train: torch.Size([608141, 32])
X_val: torch.Size([67572, 32])
Y_val: torch.Size([67572, 32])
X_train_small: torch.Size([50000, 32])
X_val_small: torch.Size([10000, 32])


In [63]:
@torch.no_grad()
def estimate_loss(model, X_data, Y_data, batch_size=32, eval_steps=30):
    model.eval()
    losses = []

    for _ in range(eval_steps):
        idx = torch.randint(0, X_data.size(0), (batch_size,), device=X_data.device)
        xb = X_data[idx]
        yb = Y_data[idx]

        _, loss = model(xb, yb)
        losses.append(loss.item())

    model.train()
    return sum(losses) / len(losses)

In [64]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

X_train_small = X_train_small.to(device)
Y_train_small = Y_train_small.to(device)
X_val_small = X_val_small.to(device)
Y_val_small = Y_val_small.to(device)

Using device: cuda


In [65]:
vocab_size = len(tokenizer.word_to_id)
block_size = 32

model = TinyGPT(
    vocab_size=vocab_size,
    block_size=block_size,
    embed_dim=192,
    num_heads=4,
    num_layers=3
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

print(sum(p.numel() for p in model.parameters()) / 1e6, "M parameters")

9.842117 M parameters


In [66]:
batch_size = 32
steps = 1000

for step in range(steps):
    idx = torch.randint(0, X_train_small.size(0), (batch_size,), device=device)
    xb = X_train_small[idx]
    yb = Y_train_small[idx]

    logits, loss = model(xb, yb)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 100 == 0:
        train_loss = estimate_loss(model, X_train_small, Y_train_small, batch_size=32, eval_steps=20)
        val_loss = estimate_loss(model, X_val_small, Y_val_small, batch_size=32, eval_steps=20)
        print(f"step {step} | train loss {train_loss:.4f} | val loss {val_loss:.4f}")

step 0 | train loss 10.0544 | val loss 10.0457
step 100 | train loss 5.7833 | val loss 5.9086
step 200 | train loss 4.9980 | val loss 5.2765
step 300 | train loss 4.3168 | val loss 5.0035
step 400 | train loss 3.8742 | val loss 4.8766
step 500 | train loss 3.5977 | val loss 4.7167
step 600 | train loss 3.3096 | val loss 4.8451
step 700 | train loss 2.9750 | val loss 4.6338
step 800 | train loss 2.7964 | val loss 4.6436
step 900 | train loss 2.6297 | val loss 4.8102


In [67]:
def generate(model, tokenizer, start_text, max_new_tokens=30):
    model.eval()
    ids = tokenizer.encode(start_text)
    ids = ids[:block_size]

    x = torch.tensor([ids], dtype=torch.long, device=device)

    for _ in range(max_new_tokens):
        x_cond = x[:, -block_size:]
        logits, _ = model(x_cond)
        logits = logits[:, -1, :]
        probs = F.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        x = torch.cat([x, next_id], dim=1)

    return tokenizer.decode(x[0].tolist())

In [68]:
print(generate(model, tokenizer, "شقة للبيع في الرياض", max_new_tokens=25))
print(generate(model, tokenizer, "فيلا للبيع في جدة", max_new_tokens=25))

<unk> للبيع في الرياض بالصور الشارع :شارعين بمدينه جده. السعر 7200682639 ريال. المساحه 140.0 متر مربع. عدد غرف النوم 3. عدد دورات المياه 3. عدد غرف المعيشه 2. المميزات:
فيلا للبيع في <unk> 130.0 3 غرف وصاله ومطبخ بسطح . ودورتين مياه . مطبخ جميع 320000 والسطح الاحجام . للتواصل ثلاجه الداخل سنوات 9 15 سنه الانترنت علي


### finetune

In [69]:
from datasets import load_dataset

ds = load_dataset("afaskar/Aqar.fm-Saudi-Real-Estate-Listings", "default")
df = ds["train"].to_pandas()

df = df.dropna(subset=[
    "category_name",
    "city",
    "district",
    "price",
    "area_sqm",
    "num_bedrooms",
    "num_bathrooms"
]).copy()

print(df.shape)

(3418, 79)


#### arabic normalization

In [70]:
import re

def normalize_arabic(text):
    text = str(text)

    text = re.sub("[إأآا]", "ا", text)
    text = re.sub("ى", "ي", text)
    text = re.sub("ة", "ه", text)

    tashkeel = re.compile(r'[\u0617-\u061A\u064B-\u0652]')
    text = re.sub(tashkeel, "", text)

    text = re.sub("ـ", "", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [71]:
import json
from collections import Counter

class SimpleTokenizer:
    def __init__(self):
        self.word_to_id = {}
        self.id_to_word = {}

    def train(self, texts, min_freq=1):
        counter = Counter()

        for text in texts:
            words = str(text).split()
            counter.update(words)

        vocab = [w for w, c in counter.items() if c >= min_freq]
        vocab = ["<pad>", "<unk>"] + sorted(vocab)

        self.word_to_id = {w: i for i, w in enumerate(vocab)}
        self.id_to_word = {i: w for w, i in self.word_to_id.items()}

    def encode(self, text):
        words = str(text).split()
        return [self.word_to_id.get(w, self.word_to_id["<unk>"]) for w in words]

    def decode(self, ids):
        return " ".join(self.id_to_word.get(i, "<unk>") for i in ids)

    def save(self, path):
        with open(path, "w", encoding="utf-8") as f:
            json.dump(self.word_to_id, f, ensure_ascii=False)

    def load(self, path):
        with open(path, "r", encoding="utf-8") as f:
            self.word_to_id = json.load(f)
        self.id_to_word = {i: w for w, i in self.word_to_id.items()}

In [72]:
def make_input_text(row):
    return normalize_arabic(
        f"{row['category_name']} في {row['district']} بمدينه {row['city']}. "
        f"السعر {int(row['price'])} ريال. "
        f"المساحه {int(row['area_sqm'])} متر مربع. "
        f"عدد غرف النوم {int(row['num_bedrooms'])}. "
        f"عدد دورات المياه {int(row['num_bathrooms'])}."
    )

In [73]:
def make_fact_output(row):
    return normalize_arabic(
        f"نوع العقار: {row['category_name']}\n"
        f"المدينه: {row['city']}\n"
        f"الحي: {row['district']}\n"
        f"السعر: {int(row['price'])} ريال\n"
        f"المساحه: {int(row['area_sqm'])} متر مربع\n"
        f"غرف النوم: {int(row['num_bedrooms'])}\n"
        f"دورات المياه: {int(row['num_bathrooms'])}"
    )

In [74]:
sft_data = []

for _, row in df.iterrows():
    sft_data.append({
        "instruction": "استخرج معلومات العقار",
        "input": make_input_text(row),
        "output": make_fact_output(row)
    })

print("Number of SFT samples:", len(sft_data))
print(sft_data[0])

Number of SFT samples: 3418
{'instruction': 'استخرج معلومات العقار', 'input': 'دور للبيع في حي النرجس بمدينه الرياض. السعر 1500000 ريال. المساحه 116 متر مربع. عدد غرف النوم 4. عدد دورات المياه 3.', 'output': 'نوع العقار: دور للبيع المدينه: الرياض الحي: حي النرجس السعر: 1500000 ريال المساحه: 116 متر مربع غرف النوم: 4 دورات المياه: 3'}


In [75]:
def format_sft_example(example):
    return (
        "التعليمات\n"
        f"{example['instruction']}\n\n"
        "المدخل\n"
        f"{example['input']}\n\n"
        "الاستجابه\n"
        f"{example['output']}"
    )

sft_texts = [format_sft_example(x) for x in sft_data]

print(sft_texts[0])

التعليمات
استخرج معلومات العقار

المدخل
دور للبيع في حي النرجس بمدينه الرياض. السعر 1500000 ريال. المساحه 116 متر مربع. عدد غرف النوم 4. عدد دورات المياه 3.

الاستجابه
نوع العقار: دور للبيع المدينه: الرياض الحي: حي النرجس السعر: 1500000 ريال المساحه: 116 متر مربع غرف النوم: 4 دورات المياه: 3


In [76]:
with open("../../data/pretrain/data.txt", "r", encoding="utf-8") as f:
    pretrain_texts = f.readlines()

pretrain_texts = [normalize_arabic(t) for t in pretrain_texts if str(t).strip()]

print(len(pretrain_texts))
print(pretrain_texts[0][:300])

6252
دور للبيع في حي النرجس بمدينه الرياض. السعر 1500000 ريال. المساحه 116.0 متر مربع. عدد غرف النوم 4. عدد دورات المياه 3. عدد غرف المعيشه 1. المميزات: مفروش: لا، مكيف: لا، مصعد: لا، مدخل سياره: نعم، مسبح: لا. الوصف: مشروع ادوار سدرا مشروع يتميز بموقعه الاستراتجي بحي النرجس جنوب طريق انس بن مالك وتصاميم


In [77]:
combined_tokenizer_texts = pretrain_texts + sft_texts

tokenizer = SimpleTokenizer()
tokenizer.train(combined_tokenizer_texts, min_freq=1)

print("Vocab size:", len(tokenizer.word_to_id))

Vocab size: 31281


In [78]:
for w in ["التعليمات", "استخرج", "معلومات", "العقار", "المدخل", "الاستجابه"]:
    print(w, w in tokenizer.word_to_id)

التعليمات True
استخرج True
معلومات True
العقار True
المدخل True
الاستجابه True


In [79]:
class PretrainDataset:
    def __init__(self, token_ids, seq_length=32):
        self.inputs = []
        self.targets = []

        for i in range(len(token_ids) - seq_length):
            x = token_ids[i:i + seq_length]
            y = token_ids[i + 1:i + seq_length + 1]
            self.inputs.append(x)
            self.targets.append(y)

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        return self.inputs[idx], self.targets[idx]

In [80]:
class PretrainDataset:
    def __init__(self, token_ids, seq_length=32):
        self.inputs = []
        self.targets = []

        for i in range(len(token_ids) - seq_length):
            x = token_ids[i:i + seq_length]
            y = token_ids[i + 1:i + seq_length + 1]
            self.inputs.append(x)
            self.targets.append(y)

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        return self.inputs[idx], self.targets[idx]

In [81]:
split_idx = int(0.9 * len(X))

X_train = X[:split_idx]
Y_train = Y[:split_idx]

X_val = X[split_idx:]
Y_val = Y[split_idx:]

X_train_small = X_train[:50000]
Y_train_small = Y_train[:50000]

X_val_small = X_val[:10000]
Y_val_small = Y_val[:10000]

print(X_train_small.shape, X_val_small.shape)

torch.Size([50000, 32]) torch.Size([10000, 32])


In [82]:
import math
import torch.nn as nn
import torch.nn.functional as F

class SelfAttentionHead(nn.Module):
    def __init__(self, embed_dim, head_dim, block_size):
        super().__init__()
        self.key = nn.Linear(embed_dim, head_dim, bias=False)
        self.query = nn.Linear(embed_dim, head_dim, bias=False)
        self.value = nn.Linear(embed_dim, head_dim, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(0.1)

    def forward(self, x):
        B, T, C = x.shape

        k = self.key(x)
        q = self.query(x)

        wei = q @ k.transpose(-2, -1) / math.sqrt(k.size(-1))
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)

        v = self.value(x)
        out = wei @ v
        return out


class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, embed_dim, block_size):
        super().__init__()
        head_dim = embed_dim // num_heads
        self.heads = nn.ModuleList([
            SelfAttentionHead(embed_dim, head_dim, block_size)
            for _ in range(num_heads)
        ])
        self.proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(0.1)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out


class FeedForward(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embed_dim, 4 * embed_dim),
            nn.ReLU(),
            nn.Linear(4 * embed_dim, embed_dim),
            nn.Dropout(0.1)
        )

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, block_size):
        super().__init__()
        self.sa = MultiHeadAttention(num_heads, embed_dim, block_size)
        self.ffwd = FeedForward(embed_dim)
        self.ln1 = nn.LayerNorm(embed_dim)
        self.ln2 = nn.LayerNorm(embed_dim)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x


class TinyGPT(nn.Module):
    def __init__(self, vocab_size, block_size, embed_dim=192, num_heads=4, num_layers=3):
        super().__init__()
        self.block_size = block_size

        self.token_embedding = nn.Embedding(vocab_size, embed_dim)
        self.position_embedding = nn.Embedding(block_size, embed_dim)

        self.blocks = nn.Sequential(*[
            TransformerBlock(embed_dim, num_heads, block_size)
            for _ in range(num_layers)
        ])

        self.ln_f = nn.LayerNorm(embed_dim)
        self.lm_head = nn.Linear(embed_dim, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.token_embedding(idx)
        pos_emb = self.position_embedding(torch.arange(T, device=idx.device))
        x = tok_emb + pos_emb

        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets, ignore_index=-100)

        return logits, loss

In [83]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

X_train_small = X_train_small.to(device)
Y_train_small = Y_train_small.to(device)
X_val_small = X_val_small.to(device)
Y_val_small = Y_val_small.to(device)

Using device: cuda


In [84]:
vocab_size = len(tokenizer.word_to_id)
block_size = 32

model = TinyGPT(
    vocab_size=vocab_size,
    block_size=block_size,
    embed_dim=192,
    num_heads=4,
    num_layers=3
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

print(sum(p.numel() for p in model.parameters()) / 1e6, "M parameters")

13.382577 M parameters


In [85]:
@torch.no_grad()
def estimate_loss(model, X_data, Y_data, batch_size=32, eval_steps=20):
    model.eval()
    losses = []

    for _ in range(eval_steps):
        idx = torch.randint(0, X_data.size(0), (batch_size,), device=X_data.device)
        xb = X_data[idx]
        yb = Y_data[idx]

        _, loss = model(xb, yb)
        losses.append(loss.item())

    model.train()
    return sum(losses) / len(losses)

In [86]:
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

batch_size = 32
steps = 500

for step in range(steps):
    idx = torch.randint(0, X_train_small.size(0), (batch_size,), device=device)
    xb = X_train_small[idx]
    yb = Y_train_small[idx]

    _, loss = model(xb, yb)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 100 == 0:
        train_loss = estimate_loss(model, X_train_small, Y_train_small)
        val_loss = estimate_loss(model, X_val_small, Y_val_small)
        print(f"pretrain step {step} | train loss {train_loss:.4f} | val loss {val_loss:.4f}")

pretrain step 0 | train loss 10.4597 | val loss 10.4560
pretrain step 100 | train loss 5.7745 | val loss 5.8446
pretrain step 200 | train loss 4.8929 | val loss 5.2845
pretrain step 300 | train loss 4.2916 | val loss 5.0120
pretrain step 400 | train loss 3.9925 | val loss 4.8932


In [87]:
class MaskedSFTDataset:
    def __init__(self, formatted_examples, tokenizer, seq_length=32):
        self.inputs = []
        self.targets = []

        for ex in formatted_examples:
            full_text = ex
            parts = full_text.split("الاستجابه\n")

            if len(parts) != 2:
                continue

            prompt = parts[0] + "الاستجابه\n"
            response = parts[1]

            prompt_ids = tokenizer.encode(prompt)
            full_ids = tokenizer.encode(full_text)

            if len(full_ids) < seq_length + 1:
                continue

            for i in range(len(full_ids) - seq_length):
                x = full_ids[i:i + seq_length]
                y = full_ids[i + 1:i + seq_length + 1]

                y_masked = y.copy()
                for j in range(len(y_masked)):
                    global_pos = i + 1 + j
                    if global_pos < len(prompt_ids):
                        y_masked[j] = -100

                self.inputs.append(x)
                self.targets.append(y_masked)

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        return self.inputs[idx], self.targets[idx]

In [88]:
masked_sft_dataset = MaskedSFTDataset(sft_texts, tokenizer, seq_length=32)

X_sft = torch.tensor(masked_sft_dataset.inputs, dtype=torch.long)
Y_sft = torch.tensor(masked_sft_dataset.targets, dtype=torch.long)

print("X_sft shape:", X_sft.shape)
print("Y_sft shape:", Y_sft.shape)

X_sft shape: torch.Size([62196, 32])
Y_sft shape: torch.Size([62196, 32])


In [89]:
split_idx = int(0.9 * len(X_sft))

X_sft_train = X_sft[:split_idx]
Y_sft_train = Y_sft[:split_idx]

X_sft_val = X_sft[split_idx:]
Y_sft_val = Y_sft[split_idx:]

X_sft_train_small = X_sft_train[:30000]
Y_sft_train_small = Y_sft_train[:30000]

X_sft_val_small = X_sft_val[:5000]
Y_sft_val_small = Y_sft_val[:5000]

X_sft_train_small = X_sft_train_small.to(device)
Y_sft_train_small = Y_sft_train_small.to(device)
X_sft_val_small = X_sft_val_small.to(device)
Y_sft_val_small = Y_sft_val_small.to(device)

print(X_sft_train_small.shape, X_sft_val_small.shape)

torch.Size([30000, 32]) torch.Size([5000, 32])


In [90]:
@torch.no_grad()
def estimate_sft_loss(model, X_data, Y_data, batch_size=16, eval_steps=20):
    model.eval()
    losses = []

    for _ in range(eval_steps):
        idx = torch.randint(0, X_data.size(0), (batch_size,), device=X_data.device)
        xb = X_data[idx]
        yb = Y_data[idx]

        _, loss = model(xb, yb)
        losses.append(loss.item())

    model.train()
    return sum(losses) / len(losses)

In [91]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

batch_size = 16
steps = 800

for step in range(steps):
    idx = torch.randint(0, X_sft_train_small.size(0), (batch_size,), device=device)
    xb = X_sft_train_small[idx]
    yb = Y_sft_train_small[idx]

    _, loss = model(xb, yb)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 50 == 0:
        train_loss = estimate_sft_loss(model, X_sft_train_small, Y_sft_train_small)
        val_loss = estimate_sft_loss(model, X_sft_val_small, Y_sft_val_small)
        print(f"sft step {step} | train loss {train_loss:.4f} | val loss {val_loss:.4f}")

sft step 0 | train loss 12.6028 | val loss 12.5823
sft step 50 | train loss 6.4231 | val loss 6.3731
sft step 100 | train loss 3.7639 | val loss 3.7912
sft step 150 | train loss 2.4920 | val loss 2.4635
sft step 200 | train loss 1.8200 | val loss 1.8690
sft step 250 | train loss 1.5384 | val loss 1.5328
sft step 300 | train loss 1.4397 | val loss 1.4007
sft step 350 | train loss 1.3135 | val loss 1.3027
sft step 400 | train loss 1.2454 | val loss 1.2311
sft step 450 | train loss 1.2159 | val loss 1.1381
sft step 500 | train loss 1.1103 | val loss 1.1838
sft step 550 | train loss 1.0923 | val loss 1.1154
sft step 600 | train loss 1.0750 | val loss 1.1007
sft step 650 | train loss 1.0091 | val loss 1.0552
sft step 700 | train loss 1.0163 | val loss 1.0110
sft step 750 | train loss 0.9972 | val loss 1.0120


In [92]:
def generate_sft(model, tokenizer, start_text, max_new_tokens=20):
    model.eval()

    ids = tokenizer.encode(start_text)
    if len(ids) == 0:
        return "لا توجد كلمات معروفه."

    x = torch.tensor([ids], dtype=torch.long, device=device)

    for _ in range(max_new_tokens):
        x_cond = x[:, -32:]
        logits, _ = model(x_cond)
        logits = logits[:, -1, :]
        next_id = torch.argmax(logits, dim=-1, keepdim=True)
        x = torch.cat([x, next_id], dim=1)

    return tokenizer.decode(x[0].tolist())

In [ ]:
prompt = """التعليمات
استخرج معلومات العقار

المدخل
شقه للبيع في حي الياسمين بمدينه الرياض. السعر 850000 ريال. المساحه 140 متر مربع. عدد غرف النوم 3. عدد دورات المياه 2.

الاستجابه
"""

print(generate_sft(model, tokenizer, prompt, max_new_tokens=12))

التعليمات استخرج معلومات العقار المدخل شقه للبيع في حي الياسمين بمدينه الرياض. السعر 850000 ريال. المساحه 140 متر مربع. عدد غرف النوم 3. عدد دورات المياه 2. الاستجابه نوع العقار: شقه للبيع المدينه: الرياض الحي: حي المهديه السعر: 800000 ريال المساحه: 200 متر مربع غرف النوم: 4 دورات


In [ ]:
prompt2 = """التعليمات
استخرج معلومات العقار

المدخل
فيلا للبيع في حي الرمال بمدينه الرياض. السعر 2200000 ريال. المساحه 300 متر مربع. عدد غرف النوم 5. عدد دورات المياه 4.

الاستجابه
"""

print(generate_sft(model, tokenizer, prompt, max_new_tokens=12))

التعليمات استخرج معلومات العقار المدخل فيلا للبيع في حي الرمال بمدينه الرياض. السعر 2200000 ريال. المساحه 300 متر مربع. عدد غرف النوم 5. عدد دورات المياه 4. الاستجابه نوع العقار: فيلا للبيع المدينه: الرياض الحي: حي المهديه السعر: 800000 ريال المساحه: 200 متر مربع غرف النوم: 4 دورات


In [95]:
torch.save(model.state_dict(), "tinygpt_real_estate_extract.pt")
tokenizer.save("tokenizer_vocab_extract.json")

print("Saved extraction model and tokenizer")

Saved extraction model and tokenizer
